In [409]:
import pandas as pd
import csv
from pathlib import Path
data_path = Path('data/')

rows = []
with open(data_path / 'data.csv', newline="") as csvFile:
    reader = csv.reader(csvFile)
    for i, row in enumerate(reader):
        if len(row) > 1:
            row = row[:-1]
        rows.append(row)
print(rows[:20])

[['Australian Bureau of Statistics'], [], ['2021 Census - selected dwelling characteristics'], ['SA2 (EN) by RNTRD Rent (weekly) Ranges by HIND Total Household Income (weekly)'], ['Counting: Dwelling Records'], [], ['Filters:'], ['Default Summation'], [], [' Braidwood'], ['HIND Total Household Income (weekly)', 'Negative income', 'Nil income', '$1-$149 ($1-$7,799)', '$150-$299 ($7,800-$15,599)', '$300-$399 ($15,600-$20,799)', '$400-$499 ($20,800-$25,999)', '$500-$649 ($26,000-$33,799)', '$650-$799 ($33,800-$41,599)', '$800-$999 ($41,600-$51,999)', '$1,000-$1,249 ($52,000-$64,999)', '$1,250-$1,499 ($65,000-$77,999)', '$1,500-$1,749 ($78,000-$90,999)', '$1,750-$1,999 ($91,000-$103,999)', '$2,000-$2,499 ($104,000-$129,999)', '$2,500-$2,999 ($130,000-$155,999)', '$3,000-$3,499 ($156,000-$181,999)', '$3,500-$3,999 ($182,000-$207,999)', '$4,000-$4,499 ($208,000-$233,999)', '$4,500-$4,999 ($234,000-$259,999)', '$5,000-$5,999 ($260,000-$311,999)', '$6,000-$7,999 ($312,000-$415,999)', '$8,000 o

In [410]:
# pad all rows to the same maximum length
max_row_len = max(len(row) for row in rows)
rows = [row + [None] * (max_row_len - len(row)) for row in rows]
raw_data = pd.DataFrame(rows)
raw_data

,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
0,Australian Bureau of Statistics,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2021 Census - selected dwelling characteristics,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,SA2 (EN) by RNTRD Rent (weekly) Ranges by HIND...,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,Counting: Dwelling Records,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81322,INFO,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
81323,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
81324,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
81325,"Copyright Commonwealth of Australia, 2025, see...",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


Now that we have a nice dataframe showing our raw data, we can begin preprocessing it. We need to extract the suburb for each wafer as well as get rid of empty rows and header rows.

In [411]:
cleaned_data = raw_data.copy()
# get rid of all rows that are entirely empty
cleaned_data = cleaned_data.dropna(how='all')
# extract the headers
headers = raw_data.loc[10, :]
# exclude the ABS information (not data for our purposes)
cleaned_data = cleaned_data.loc[9:, :]
cleaned_data

,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
9,Braidwood,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
10,HIND Total Household Income (weekly),Negative income,Nil income,"$1-$149 ($1-$7,799)","$150-$299 ($7,800-$15,599)","$300-$399 ($15,600-$20,799)","$400-$499 ($20,800-$25,999)","$500-$649 ($26,000-$33,799)","$650-$799 ($33,800-$41,599)","$800-$999 ($41,600-$51,999)",...,"$3,500-$3,999 ($182,000-$207,999)","$4,000-$4,499 ($208,000-$233,999)","$4,500-$4,999 ($234,000-$259,999)","$5,000-$5,999 ($260,000-$311,999)","$6,000-$7,999 ($312,000-$415,999)","$8,000 or more ($416,000 or more)",Partial income stated,All incomes not stated,Not applicable,Total
11,RNTRD Rent (weekly) Ranges,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
12,$1-$74,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
13,$75-$99,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81318,Total,35381,135890,61662,115443,207226,518470,383480,545103,559265,...,366591,206641,383541,262326,229548,88070,481636,187188,1439142,10875248
81320,"Dataset: Census of Population and Housing, 202...",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
81322,INFO,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
81325,"Copyright Commonwealth of Australia, 2025, see...",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [412]:
rent_weekly_title = cleaned_data.loc[11]
cleaned_data.rename_axis(rent_weekly_title[0], inplace=True)
cleaned_data = cleaned_data.loc[cleaned_data.loc[:, 0] != rent_weekly_title[0]]
cleaned_data

,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
RNTRD Rent (weekly) Ranges,,,,,,,,,,,,,,,,,,,,,
9,Braidwood,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
10,HIND Total Household Income (weekly),Negative income,Nil income,"$1-$149 ($1-$7,799)","$150-$299 ($7,800-$15,599)","$300-$399 ($15,600-$20,799)","$400-$499 ($20,800-$25,999)","$500-$649 ($26,000-$33,799)","$650-$799 ($33,800-$41,599)","$800-$999 ($41,600-$51,999)",...,"$3,500-$3,999 ($182,000-$207,999)","$4,000-$4,499 ($208,000-$233,999)","$4,500-$4,999 ($234,000-$259,999)","$5,000-$5,999 ($260,000-$311,999)","$6,000-$7,999 ($312,000-$415,999)","$8,000 or more ($416,000 or more)",Partial income stated,All incomes not stated,Not applicable,Total
12,$1-$74,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
13,$75-$99,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
14,$100-$124,0,0,0,0,0,5,0,0,0,...,0,0,0,0,0,0,0,0,0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81318,Total,35381,135890,61662,115443,207226,518470,383480,545103,559265,...,366591,206641,383541,262326,229548,88070,481636,187188,1439142,10875248
81320,"Dataset: Census of Population and Housing, 202...",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
81322,INFO,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [413]:
suburb_names = cleaned_data.loc[cleaned_data.isna().sum(axis=1) > 0, 0]
suburb_names = suburb_names[~suburb_names.isin(['Total'])]
suburb_names

RNTRD Rent (weekly) Ranges
9                                                Braidwood
42                                                 Karabar
75                                              Queanbeyan
108                                      Queanbeyan - East
141                        Queanbeyan West - Jerrabomberra
                               ...                        
81288                                                Total
81320    Dataset: Census of Population and Housing, 202...
81322                                                 INFO
81325    Copyright Commonwealth of Australia, 2025, see...
81326    ABS data licensed under Creative Commons, see ...
Name: 0, Length: 2468, dtype: object

In [414]:
cleaned_data.loc[suburb_names.index, ['Suburb']] = suburb_names
# forward fill downwards, filling each cell in the Suburb column with the suburb for each data point.
cleaned_data.loc[:, 'Suburb'] = cleaned_data.loc[:, 'Suburb'].ffill()
# remove all the rows that have empty cells. These are not actual data rows. They're just typical ABS file info.
cleaned_data = cleaned_data.dropna(how='any')
# finally, we delete the rows which contain headers
cleaned_data = cleaned_data[cleaned_data.iloc[:, 1] != 'Negative income']
cleaned_data

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,Suburb
RNTRD Rent (weekly) Ranges,,,,,,,,,,,,,,,,,,,,,
12,$1-$74,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,Braidwood
13,$75-$99,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Braidwood
14,$100-$124,0,0,0,0,0,5,0,0,0,...,0,0,0,0,0,0,0,0,10,Braidwood
15,$125-$149,0,0,0,0,0,0,5,0,0,...,0,0,0,0,0,0,0,0,5,Braidwood
16,$150-$174,0,0,0,0,0,0,7,5,0,...,0,0,0,0,0,0,3,0,20,Braidwood
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81314,$850-$949,57,310,78,124,128,183,205,339,435,...,1420,4130,2952,3127,887,2120,146,20,28378,Total
81315,$950 and over,170,748,185,216,285,387,516,672,892,...,1834,8297,5539,7449,3682,3876,483,59,53785,Total
81316,Not stated,2150,5375,1490,3159,4886,6629,4918,5653,6371,...,1123,2742,1139,947,497,10403,13789,505,114130,Total


We can now drop the first column of the dataframe since it's just the row metadata.

In [415]:
# make the dataframe's index equal to the rent ranges
cleaned_data.index = cleaned_data.loc[:, 0]
# rename the index because it got changed from the previous line
cleaned_data.rename_axis(rent_weekly_title[0], inplace=True)
# drop the first column
cleaned_data = cleaned_data.loc[:, 1:]
# make the columns equal to the desired header
cleaned_data.columns = list(headers[1:]) + ['Suburb']
cleaned_data

,Negative income,Nil income,"$1-$149 ($1-$7,799)","$150-$299 ($7,800-$15,599)","$300-$399 ($15,600-$20,799)","$400-$499 ($20,800-$25,999)","$500-$649 ($26,000-$33,799)","$650-$799 ($33,800-$41,599)","$800-$999 ($41,600-$51,999)","$1,000-$1,249 ($52,000-$64,999)",...,"$4,000-$4,499 ($208,000-$233,999)","$4,500-$4,999 ($234,000-$259,999)","$5,000-$5,999 ($260,000-$311,999)","$6,000-$7,999 ($312,000-$415,999)","$8,000 or more ($416,000 or more)",Partial income stated,All incomes not stated,Not applicable,Total,Suburb
RNTRD Rent (weekly) Ranges,,,,,,,,,,,,,,,,,,,,,
$1-$74,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,Braidwood
$75-$99,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Braidwood
$100-$124,0,0,0,0,0,5,0,0,0,0,...,0,0,0,0,0,0,0,0,10,Braidwood
$125-$149,0,0,0,0,0,0,5,0,0,0,...,0,0,0,0,0,0,0,0,5,Braidwood
$150-$174,0,0,0,0,0,0,7,5,0,0,...,0,0,0,0,0,0,3,0,20,Braidwood
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
$850-$949,57,310,78,124,128,183,205,339,435,654,...,1420,4130,2952,3127,887,2120,146,20,28378,Total
$950 and over,170,748,185,216,285,387,516,672,892,1256,...,1834,8297,5539,7449,3682,3876,483,59,53785,Total
Not stated,2150,5375,1490,3159,4886,6629,4918,5653,6371,6956,...,1123,2742,1139,947,497,10403,13789,505,114130,Total


Quick sanity check. Any group-by by suburb we have should be equal in length to the number of wafers in our file (2464).

In [416]:
cleaned_data.groupby('Suburb')['Negative income'].sum()

Suburb
ACT - South West          0000000000000000000000000000
APY Lands                 0000000000000000000000000000
Abbotsford                0000000000000000000000000036
Aberfoyle Park            0000000000000000000000000304
Acacia Gardens            0000000000000000000000000000
                                     ...              
Young Surrounds         000000000000000000000000001918
Youngtown - Relbia        0000000000000000000000000055
Yuendumu - Anmatjere      0000000000000000000000000000
Zetland                 000000000000000000300644001426
Zillmere                 00000000030000000000000000010
Name: Negative income, Length: 2464, dtype: object

In [417]:
# save the cleaned data to a new CSV file for use by the other notebooks.
cleaned_data.to_csv('data/cleaned_data.csv')